# Sesión 2 · Robustez y evaluación

**Prácticas de *LLMs aplicados a Finanzas* · MIAX · jueves 17 de septiembre**

## Dónde lo dejamos

El día 10 construisteis un agente que funciona: cuatro herramientas, un bucle
ReAct escrito a mano y `create_agent` haciendo lo mismo en cinco líneas.
Terminamos con una celda que entraba en bucle infinito y no la arreglamos.

Hoy no vamos a añadirle capacidades. Vamos a **medirlo y a ponerle límites**,
que es lo que separa una demo de algo que se puede defender delante de
alguien. La diferencia entre las dos cosas es una tabla de números, y esa
tabla es la mitad de vuestro entregable del 24.

## El orden del día lo pone el diagnóstico

En quince minutos vais a ejecutar vuestro agente contra cinco preguntas
elegidas para que falle. Los cinco fallos que salgan son el índice de la
sesión:

| Fallo | Dónde se cierra |
| --- | --- |
| No encontró nada relevante | §1 · retrieval |
| Recuperó lo que no era | §1 · retrieval |
| No supo comparar dos ejercicios | §2 · estado |
| Se inventó la cifra | §3 · guardrails |
| Usó la herramienta equivocada | §3 y §4 · trayectoria |

## Lo que se abre hoy

`search_filings` era una caja negra. Dentro había cuatro decisiones —troceado,
*embeddings*, índice, top-*k*— y cada una se puede hacer mejor o peor. Hoy se
abre, se mide lo que hace y se arregla tres veces, midiendo después de cada
arreglo.

**Que el orden de esa frase se lea bien: medir, arreglar, volver a medir.** Un
arreglo sin medición previa no es una mejora, es una corazonada. Vais a ver
hoy un arreglo razonable, defendible y recomendado por la literatura que en
este corpus **no sirve de nada**, y solo se sabe porque se mide.

## Cómo se usa este notebook

- **Se ejecuta de arriba abajo, sin saltos.** Cada sección termina con
  `assert`.
- **Ninguna clave está escrita aquí dentro.** Se piden por `getpass`.
- Lo que puede fallar por red está en `try/except` y degrada. Nada debería
  parar la clase.

Junto al notebook necesitáis `miax_s2.py`, `golden_set.jsonl` y los dos ZIP
del corpus. Si traéis vuestro agente de la semana pasada, traedlo también: la
celda 5 lo usa si lo encuentra.

In [ ]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Las mismas de la sesión 1: si ya las tenéis, esto tarda segundos.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2
print("Instalación terminada.")

In [2]:
# Claves. Igual que la semana pasada: del entorno, y si no, por teclado.
import os
import getpass
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())


def pedir_clave(nombre: str, donde: str) -> bool:
    """Deja `nombre` en el entorno si se puede. Devuelve si hay clave."""
    if os.environ.get(nombre):
        print(f"{nombre}: ya estaba en el entorno.")
        return True
    try:
        valor = getpass.getpass(f"{nombre} (se saca en {donde}): ").strip()
    except Exception:                      # sin terminal interactiva
        valor = ""
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: guardada en el entorno de esta sesión.")
        return True
    print(f"{nombre}: sin clave. Las celdas que llaman al modelo no van a "
          f"funcionar; las de retrieval y medición, sí.")
    return False


HAY_CLAVE = pedir_clave("OPENROUTER_API_KEY", "openrouter.ai/keys")

OPENROUTER_API_KEY: ya estaba en el entorno.


In [3]:
# %% Corpus e indice  --------------------------------
# Los dos ZIP os los pasamos nosotros (Drive compartido, aula virtual o el
# panel de ficheros de Colab): son 5,6 MB entre los dos. Nada de descargar
# de EDGAR en vivo, que con treinta cuadernos a la vez acaba en bloqueo.
#
# Si los teneis en Drive:
#     from google.colab import drive; drive.mount("/content/drive")
# y anadid la carpeta a CANDIDATOS.
import hashlib, pathlib, sys, zipfile

# Dentro del repositorio el kernel puede arrancar en la raíz o en Clase_2/:
# se localiza la raíz y se usan rutas absolutas. En Colab no hay raíz y todo
# funciona como antes.
RAIZ = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "Clase_2" / "miax_s2.py").is_file()), None)
if RAIZ is not None:
    for carpeta in (RAIZ, RAIZ / "Clase_2"):
        if str(carpeta) not in sys.path:
            sys.path.insert(0, str(carpeta))

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = RAIZ / "corpus" if RAIZ is not None else pathlib.Path("corpus")

CANDIDATOS = ([RAIZ, RAIZ / "dataset"] if RAIZ is not None else []) + [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
    pathlib.Path("/content/dataset"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


# Si el corpus ya está extraído no se vuelve a descomprimir encima: basta
# con verificarlo contra los manifiestos más abajo.
NECESARIOS = ["chunks.jsonl", "secciones.jsonl", "xbrl_facts.parquet",
              "indice/corpus.faiss", "indice/chunks_meta.parquet"]
YA_EXTRAIDO = all((DESTINO / n).is_file() for n in NECESARIOS)

try:
    for nombre, esperado in ([] if YA_EXTRAIDO else PAQUETES):
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")

# miax_s2 busca el corpus en rutas relativas: se le indica dónde está.
import miax_s2
if DESTINO.resolve() not in [pathlib.Path(c).resolve() for c in miax_s2.CANDIDATOS_CORPUS]:
    miax_s2.CANDIDATOS_CORPUS.insert(0, DESTINO)


Corpus e indice verificados en /Users/emiliosanchez/TallerB5-T5/Taller-B5-T5-NPL/corpus
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  chunks.jsonl                    3.80 MB
  derivado/etiquetas.parquet      0.03 MB
  derivado/etiquetas.sha256       0.00 MB
  indice/MANIFEST.md              0.00 MB
  indice/chunks_meta.parquet      1.48 MB
  indice/corpus.faiss             2.69 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


In [4]:
# Vuestro agente, o uno de repuesto.
#
# Esta celda intenta tres cosas, en este orden:
#   1. `from agente.interfaz import responder` — vuestro repositorio, tal y
#      como lo pide el §6 del enunciado.
#   2. `agente` ya definido en esta sesión de Colab.
#   3. `miax_s2.baseline()` — el del día 10, montado de repuesto.
#
# El tercero existe para que nadie se quede fuera hoy. No es un atajo: quien
# lo use empieza la sesión sin conocer su propio código, y el día 24 defiende
# el suyo.
import sys
sys.path.append('/content')
import json
from pathlib import Path

import miax_s2

MODELO = os.getenv("MODELO_10K", "openrouter:google/gemini-3.8-flash")

AQUI = RAIZ / "Clase_2" if RAIZ is not None else Path(".")
RUTA_GOLDEN = AQUI / "golden_set.jsonl"
if not RUTA_GOLDEN.is_file():                        # respaldo de la S1
    RUTA_GOLDEN = (RAIZ / "Material_Clase" if RAIZ is not None else Path(".")) / "golden_set_ejemplo.jsonl"
golden = [json.loads(l) for l in open(RUTA_GOLDEN, encoding="utf-8")
          if l.strip()]
print(f"{RUTA_GOLDEN.name}: {len(golden)} preguntas "
      f"({sum(1 for g in golden if g.get('ancla_texto'))} con ancla)")

agente = None
origen = None
try:
    if not HAY_CLAVE:
        raise RuntimeError("sin clave")
    # `responder` del repositorio devuelve sólo la respuesta estructurada (la
    # firma del holdout). Este notebook necesita el resultado completo
    # (mensajes, latencia y coste) y un thread_id: eso lo da `ejecutar`.
    from agente.interfaz import ejecutar

    def responder(pregunta, thread_id=None):
        return ejecutar(pregunta, thread_id=thread_id)
    origen = "vuestro repositorio (agente/interfaz.py)"
except Exception:
    if HAY_CLAVE:
        agente = miax_s2.baseline(MODELO)
        origen = "miax_s2.baseline() — el de repuesto"

        def responder(pregunta, thread_id=None):
            """Firma del §6 del enunciado: la misma que ejecuta el día 24."""
            resultado, segundos = miax_s2.cronometrar(
                agente.invoke,
                {"messages": [{"role": "user", "content": pregunta}]},
                config={"configurable":
                        {"thread_id": thread_id or "s2"}},
            )
            return {**resultado,
                    "coste_usd": miax_s2.coste_de(resultado, MODELO),
                    "latencia_s": segundos}

print("Agente:", origen or "NINGUNO (sin clave)")

golden_set.jsonl: 20 preguntas (13 con ancla)


Agente: vuestro repositorio (agente/interfaz.py)


In [5]:
# Las cinco preguntas duras. Sin comentarios: mirad qué sale.
#
# No son preguntas rebuscadas. Son cinco del golden set oficial elegidas
# porque cada una rompe el agente por un sitio distinto, y las cinco se
# vuelven a medir al final de la sesión.
duras = miax_s2.preguntas_duras(golden)

for d in duras:
    print("=" * 78)
    print(f"[{d['id']} · {d['fallo']}] {d['pregunta']}")
    print(f"  esperado: {d['respuesta_esperada'][:100]}")
    if origen is None:
        print("  (sin clave: no se ejecuta)")
        continue
    try:
        r = responder(d["pregunta"], thread_id=f"duras-{d['id']}")
        miax_s2.pretty_trace(r)
        print(f"  [{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
    except Exception as e:
        print(f"  FALLÓ con excepción: {type(e).__name__}: {e}")

[of-006 · No encontró nada relevante] ¿Qué porcentaje de los ingresos consolidados de Amazon aportó el segmento internacional en 2025, según el apartado de riesgo de mercado de su 10-K?
  esperado: El 23 % de los ingresos consolidados.


/Users/emiliosanchez/TallerB5-T5/Taller-B5-T5-NPL/TallerB5T5_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9492.83it/s]

  FALLÓ con excepción: RemoteProtocolError: peer closed connection without sending complete message body (incomplete chunked read)
[of-002 · Recuperó lo que no era] ¿Qué riesgo de seguridad asocia Microsoft en FY2025 al uso creciente de modelos de IA generativa en sus sistemas internos?
  esperado: Que puede abrir nuevas superficies de ataque o dar métodos nuevos a los adversarios.


...

OpenRouter: fallo de conexión; un reintento. El coste previo queda desconocido.


Límite de 150 s alcanzado: se cierra la pregunta sin respuesta.


  1. list_available()
       -> AAPL · Apple Inc. FY2024: Items 1A, 7, 7A, 8 AAPL · Apple Inc. FY2025: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2024: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2025: Items 1A, 7, 7A, 8 GOOGL · Alphabet Inc. FY2024: It…

  respuesta: Sin respuesta verificada dentro del límite de 150 s.
  fuente: ninguna · cita: None
  FALLÓ con excepción: TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
[of-019 · Usó la herramienta equivocada] ¿Cómo cambió el beneficio bruto de Apple entre FY2024 y FY2025, y qué dice la dirección del margen bruto de productos?
  esperado: El beneficio bruto subió de 180.683 millones de dólares a 195.201 millones de dólares (+8,0 %), pero


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


  1. list_available()
       -> AAPL · Apple Inc. FY2024: Items 1A, 7, 7A, 8 AAPL · Apple Inc. FY2025: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2024: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2025: Items 1A, 7, 7A, 8 GOOGL · Alphabet Inc. FY2024: It…
  2. get_xbrl_fact(concept='GrossProfit', ticker='AAPL', fiscal_year=2024)
  3. get_xbrl_fact(fiscal_year=2025, concept='GrossProfit', ticker='AAPL')
       -> AAPL FY2024 · GrossProfit = 180683000000.0 USD (cierre 2024-09-28, 10-K)
       -> AAPL FY2025 · GrossProfit = 195201000000.0 USD (cierre 2025-09-27, 10-K)
  4. search_filings(item='7', ticker='AAPL', fiscal_year=2025, query='Products gross margin percentage year over year')
       -> [AAPL-2025-7-0004] AAPL FY2025 Item 7 · Gross Margin (puesto 1 de 5) Gross Margin  Products and Services gross margin and gross margin percentage for 2025, 2024 and 2023 were as follows (dollars in millions):  2025		2024…
  5. RespuestaFinanciera(cita='Products gross margin percentage decreased duri

## Los cinco fallos, clasificados

Esto es lo que acabáis de ver, con el nombre técnico de cada cosa y la celda
donde se cierra. Copiadlo en la pizarra mental: la sesión entera va de bajar
por esta tabla.

| Fallo | Qué está pasando de verdad | Se cierra en |
| --- | --- | --- |
| **No encontró nada relevante** | La respuesta está en el Item 7A: 37 fragmentos de 1.749. Nadie escribe «7A» en su pregunta | §1, celda 11 |
| **Recuperó lo que no era** | Los 10-K repiten factores de riesgo casi literales entre ejercicios. Sin filtrar por año, lo más parecido puede ser el año equivocado | §1, celda 11 |
| **No supo comparar dos ejercicios** | Una sola pasada de recuperación no descompone una pregunta en dos | §2, celda 18 |
| **Se inventó la cifra** | El concepto US-GAAP que usa una compañía no vale para otra. Cuando no lo encuentra, rellena | §3, celda 23 |
| **Usó la herramienta equivocada** | Leyó el número de la prosa en vez de consultarlo. Acierta hoy y falla el día que la tabla venga partida | §3 y §4 |

Fijaos en que los dos primeros son **el mismo problema**: el retrieval no sabe
de qué documento le están hablando. Y es el más barato de arreglar, que es por
donde vamos a empezar.

> **La trampa del cuarto.** «Se inventó la cifra» suena a alucinación del
> modelo y casi nunca lo es. Alphabet etiqueta `Revenues`; Apple,
> `RevenueFromContractWithCustomerExcludingAssessedTax`. El modelo pide el
> concepto que le funcionó con otra compañía, la herramienta le dice que no
> está, y ahí es donde decide rellenar. El fallo empieza en el dato, no en el
> modelo.

## Qué había dentro de la caja negra

Esto es todo lo que hacía `search_filings` la semana pasada:

```text
 1  # Una vez, al construir el corpus:
 2  para cada seccion de cada 10-K:
 3      partir por encabezados
 4      agrupar en ventanas de ~500 tokens con solape de 80
 5      guardar cada ventana con su (ticker, fiscal_year, item, inicio, fin)
 6
 7  vectores = modelo_bge.encode(textos_de_los_fragmentos)   # 384 dimensiones
 8  normalizar(vectores)
 9  indice = faiss.IndexFlatIP(384)      # producto interno = coseno
10  indice.add(vectores)                 # 1.749 vectores
11
12  # En cada búsqueda:
13  v = modelo_bge.encode("Represent this sentence for...: " + consulta)
14  normalizar(v)
15  puntuaciones, posiciones = indice.search(v, k)
16  devolver los fragmentos de esas posiciones
```

Dieciséis líneas. No hay nada más, y conviene decirlo en voz alta: el 90 % de
los sistemas RAG en producción son exactamente esto, y la diferencia entre uno
que funciona y uno que no está **en las cuatro decisiones de las líneas 3, 4,
7 y 15**, no en el framework.

Las cuatro decisiones, y lo que se puede hacer mal en cada una:

| Línea | Decisión | Cómo se rompe en un 10-K |
| --- | --- | --- |
| 3-4 | **Troceado** | Una tabla de resultados partida por la mitad deja las cifras sin encabezado |
| 7 | **Modelo de *embeddings*** | Es monolingüe inglés. Vuestras preguntas están en español |
| 9 | **Índice** | `IndexFlatIP` compara con todo. Correcto y no escala; con 1.749 vectores da igual |
| 15 | **Top-*k* y filtros** | Devuelve los *k* más parecidos de TODO el corpus, mezclando compañías y ejercicios |

El código real está en
[`data_source/chunks.py`](https://github.com/) si alguien quiere mirarlo, pero
lleva encima aritmética de bytes para calcular los desplazamientos de cada
fragmento: eso es maquinaria de trazabilidad, no la idea.

> **La línea 13 es el fallo silencioso favorito del curso.** BGE exige ese
> prefijo en la consulta y **no** en los fragmentos indexados. Si lo omitís no
> pasa nada: no hay error, no hay aviso, simplemente recuperáis peor. Está
> documentado en `indice/MANIFEST.md`, que es exactamente el sitio donde nadie
> mira.

In [6]:
# El troceado, visto con los datos y no con el código.
#
# `contiene_tabla` marca los fragmentos donde el troceador partió una tabla.
# No es un caso raro: en este corpus son casi la mitad.
import pandas as pd

secciones, chunks = miax_s2.cargar_corpus()
c = pd.DataFrame([{k: v for k, v in x.items() if k != "texto"}
                  for x in chunks])

resumen = c.groupby("item").agg(
    fragmentos=("chunk_id", "size"),
    con_tabla=("contiene_tabla", "sum"),
    tokens_medios=("n_tokens", "mean"),
).round(0).astype(int)
resumen["% con tabla"] = (100 * resumen.con_tabla
                          / resumen.fragmentos).round(0).astype(int)
print(resumen.to_string())
print(f"\nTotal: {int(c.contiene_tabla.sum())} de {len(c)} fragmentos "
      f"({100*c.contiene_tabla.mean():.0f} %) llevan una tabla dentro.")

# Y una partida por la mitad, que es lo que hay que ver.
#
# El Item 8 son los estados financieros: ahí el texto ES una tabla, y una
# ventana de 500 tokens la corta por donde le toca. Buscamos dos fragmentos
# CONSECUTIVOS que lleven tabla los dos: la frontera entre ellos es el corte.
por_seccion = {}
for x in chunks:
    por_seccion.setdefault(
        (x["ticker"], x["fiscal_year"], x["item"]), []).append(x)

corte = None
for clave, lista in por_seccion.items():
    if clave[2] != "8":
        continue
    lista.sort(key=lambda x: x["posicion"])
    for a, b in zip(lista, lista[1:]):
        if a["contiene_tabla"] and b["contiene_tabla"]:
            corte = (a, b)
            break
    if corte:
        break

a, b = corte
print(f"\n--- final de {a['chunk_id']} ---")
print(a["texto"][-450:])
print(f"\n>>> AQUÍ CORTA EL TROCEADOR <<<\n")
print(f"--- principio de {b['chunk_id']} ---")
print(b["texto"][:450])

# Lo que le llega al modelo si recupera solo el segundo: filas de números sin
# el encabezado que dice qué son ni de qué ejercicio. Y el encabezado estaba
# en el fragmento anterior, que no ha recuperado porque no se parecía tanto
# a la pregunta.

      fragmentos  con_tabla  tokens_medios  % con tabla
item                                                   
1A           533         34            443            6
7            307        120            371           39
7A            37         18            345           49
8            872        549            390           63

Total: 721 de 1749 fragmentos (41 %) llevan una tabla dentro.

--- final de NVDA-2024-8-0006 ---
ax expense (benefit)	4,058			( 187 )			189
Net income	$	29,760			$	4,368			$	9,752
Net income per share:
Basic	$	12.05			$	1.76			$	3.91
Diluted	$	11.93			$	1.74			$	3.85
Weighted average shares used in per share computation:
Basic	2,469			2,487			2,496
Diluted	2,494			2,507			2,535

See accompanying notes to the consolidated financial statements.

NVIDIA Corporation and Subsidiaries

Consolidated Statements of Comprehensive Income

(In millions)

>>> AQUÍ CORTA EL TROCEADOR <<<

--- principio de NVDA-2024-8-0007 ---
Year Ended
Jan 28, 2024		Jan 29, 2023		Jan 

In [7]:
# La medición del baseline. Esto es lo que hay que hacer ANTES de arreglar
# nada, y es la parte que casi nadie hace.
#
# recall@k: de las preguntas cuya respuesta está en una frase concreta del
# informe, ¿en cuántas aparece esa frase entre los k fragmentos devueltos?
#
# La verdad NO es un chunk_id: es el `ancla_texto`, una frase literal del
# informe. Si fuese un chunk_id, en cuanto cambiarais la ventana o el solape
# todos los identificadores serían otros y la métrica se quedaría sin
# referencia — y el grupo que mejora el troceado saldría penalizado por
# haberlo mejorado.
K = 5
con_ancla = [g for g in golden if g.get("ancla_texto")]
indice, meta, _ = miax_s2.cargar_indice()
print(f"{len(con_ancla)} preguntas con ancla · {indice.ntotal} vectores")


def denso_plano(consulta: str, k: int = K) -> list[dict]:
    """Lo que hacía `search_filings` el día 10: los k más parecidos de TODO
    el corpus, sin mirar de qué compañía ni de qué ejercicio son."""
    puntuaciones, posiciones = indice.search(miax_s2.codificar([consulta]), k)
    return [miax_s2.fila_a_fragmento(meta.iloc[int(i)], s)
            for s, i in zip(puntuaciones[0], posiciones[0])]


def medir(buscar, etiqueta: str, k: int = K) -> float:
    """recall@k de una función de búsqueda sobre el golden set.

    El `or []` no es paranoia: una función de búsqueda a medio escribir
    devuelve None, y entonces esto tiene que dar 0 % y seguir, no un
    traceback a mitad de un ejercicio.
    """
    recuperados = {g["id"]: (buscar(g) or []) for g in con_ancla}
    valor = miax_s2.recall_en_k(con_ancla, recuperados)
    fallan = [g["id"] for g in con_ancla
              if not miax_s2.acierta(g, recuperados[g["id"]])]
    print(f"{etiqueta:34s} recall@{k} = {valor:6.1%}   fallan: {fallan}")
    return valor


RESULTADOS = {}
RESULTADOS["1 · denso plano"] = medir(
    lambda g: denso_plano(g["pregunta"]), "denso plano (como el día 10)")

# Y por cuánto fallan. recall@5 dice sí o no; esto dice si el ancla estaba en
# el puesto 7 o en el 1.400, que no es el mismo problema ni tiene el mismo
# arreglo.
print("\nPuesto del ancla buscando en todo el corpus:")
for g in con_ancla:
    todos = denso_plano(g["pregunta"], k=indice.ntotal)
    puesto = miax_s2.posicion_del_ancla(g, todos)
    print(f"  {g['id']}  {g['ticker']} FY{g['fiscal_year']} "
          f"item {g['item_esperado']:>2}  ->  puesto {puesto} de {len(todos)}")

13 preguntas con ancla · 1749 vectores


denso plano (como el día 10)       recall@5 =  30.8%   fallan: ['of-001', 'of-004', 'of-005', 'of-006', 'of-015', 'of-016', 'of-018', 'of-019', 'of-020']

Puesto del ancla buscando en todo el corpus:
  of-001  NVDA FY2025 item 1A  ->  puesto 217 de 1749
  of-002  MSFT FY2025 item 1A  ->  puesto 5 de 1749
  of-003  META FY2025 item 1A  ->  puesto 2 de 1749


  of-004  AAPL FY2025 item 1A  ->  puesto 1178 de 1749
  of-005  GOOGL FY2025 item 1A  ->  puesto 115 de 1749
  of-006  AMZN FY2025 item 7A  ->  puesto 27 de 1749
  of-014  MSFT FY2025 item  7  ->  puesto 1 de 1749


  of-015  NVDA FY2025 item  7  ->  puesto 166 de 1749
  of-016  META FY2025 item  7  ->  puesto 307 de 1749
  of-017  AMZN FY2025 item  7  ->  puesto 1 de 1749
  of-018  GOOGL FY2025 item  7  ->  puesto 1423 de 1749


  of-019  AAPL FY2025 item  7  ->  puesto 1157 de 1749
  of-020  META FY2025 item  7  ->  puesto 9 de 1749


CODIGO CLASE

In [8]:
# Apuntes de clase: buscar a mano en el índice FAISS.
indice, meta, m2 = miax_s2.cargar_indice()
pregunta = "beneficio neto microsoft 2025"
# codificar() añade el prefijo de consulta de BGE; m2.encode() a secas recupera peor.
intento = indice.search(miax_s2.codificar([pregunta]), 5)
# Devuelve (puntuaciones, posiciones): la similitud de cada vector encontrado
# y su fila en `meta`.
intento

(array([[0.69224906, 0.68732494, 0.6845176 , 0.67489   , 0.6621643 ]],
       dtype=float32),
 array([[1344,  470,  696,  567,  432]]))

In [9]:
# Las filas de `meta` que ha devuelto la búsqueda. El texto es texto: no se
# puede convertir a int(); las cifras salen del XBRL.
meta.iloc[intento[1][0]][["chunk_id", "ticker", "fiscal_year", "item"]]

,chunk_id,ticker,fiscal_year,item
1344,META-2025-7-0001,META,2025,7
470,MSFT-2025-7-0001,MSFT,2025,7
696,AAPL-2025-7-0002,AAPL,2025,7
567,MSFT-2025-8-0071,MSFT,2025,8
432,MSFT-2024-8-0072,MSFT,2024,8


In [10]:
# ARREGLO 1 (8 min) — Filtrar por metadatos.
#
# El más barato de los tres y el que más devuelve. Cada fragmento sabe de qué
# compañía, de qué ejercicio y de qué sección es; el golden set también. No
# usarlo es tirar información que ya está sobre la mesa.
#
# Fijaos en el orden: se busca en TODO el índice y se filtra después. Con
# 1.749 vectores eso es instantáneo. Con un corpus de verdad habría que
# filtrar antes, y esa es una conversación distinta.


def con_filtros(consulta: str, ticker=None, fiscal_year=None, item=None,
                k: int = K) -> list[dict]:
    """Igual que `denso_plano`, pero descartando lo que no cuadra."""
    # TODO (alumno): el cuerpo.
    #
    #   1. Buscad sobre TODO el índice, no sobre k:
    #      indice.search(miax_s2.codificar([consulta]), indice.ntotal) HECHO EN CELDAS ANTERIORES
    puntuaciones, posiciones = indice.search(miax_s2.codificar([consulta]), indice.ntotal)

    #   2. Recorred los resultados en orden de puntuación. `meta.iloc[i]` os
    #      da la fila del fragmento i, con sus columnas ticker, fiscal_year
    #      e item.
    resultado = []
    for puntuacion, posicion in zip(puntuaciones[0], posiciones[0]):
        fila = meta.iloc[posicion] #Cuando hablamos de fila es meta.iloc[posicion] que es la fila del dataframe meta que contiene los metadatos del fragmento correspondiente a la posición obtenida en la búsqueda.
        if ticker and fila.ticker != ticker:
            continue
        if fiscal_year is not None and int(fila.fiscal_year) != fiscal_year:
            continue
        # fila["item"], no fila.item: en pandas, Series.item es un método y
        # la comparación descartaba todos los fragmentos sin dar error.
        if item is not None and fila["item"] != item:
            continue

        # `puntuacion` y `posicion` ya son un número cada una (vienen del zip):
        # no se indexan otra vez. Se guarda sólo el fragmento en el formato de
        # la métrica; su texto ya va dentro, en fragmento["texto"].
        resultado.append(miax_s2.fila_a_fragmento(fila, puntuacion))
        if len(resultado) >= k:
            break
    return resultado
    #   3. Descartad el que no cuadre con los filtros que NO sean None.
    #      Ojo con `fiscal_year`: en el índice es un entero.
    #   4. Parad al llegar a k y devolved la lista.
    #      `miax_s2.fila_a_fragmento(fila, puntuacion)` os da el dict con el
    #      formato que espera la métrica.
    ...


RESULTADOS["2 · + filtro de metadatos"] = medir(
    lambda g: con_filtros(g["pregunta"], g["ticker"], g["fiscal_year"],
                          g["item_esperado"]),
    "+ filtro ticker/ejercicio/item")

# El matiz que hay que decir en voz alta: en producción nadie os da el ticker
# y el item en una tabla. Los tiene que sacar el agente de la pregunta y
# pasarlos como argumentos de `search_filings`.

+ filtro ticker/ejercicio/item     recall@5 =  46.2%   fallan: ['of-001', 'of-004', 'of-005', 'of-015', 'of-016', 'of-018', 'of-019']


In [11]:
# El texto del mejor fragmento.
print(meta.iloc[intento[1][0][0]].texto[:800])

Executive Overview of Full Year 2025 Results

Our mission is to build the future of human connection and the technology that makes it possible.

Our financial results and key Family metrics for 2025 are set forth below. Total revenue for 2025 was $200.97 billion, an increase of 22% compared to 2024, due to an increase in advertising revenue. Ad impressions delivered across our Family of Apps in 2025 increased 12% year-over-year, and our average price per ad increased 9% year-over-year.

Income from operations for 2025 was $83.28 billion, an increase of $13.90 billion, or 20%, compared to 2024, driven by an increase in advertising revenue, partially offset by an increase in costs and expenses. The increase in costs and expenses was mainly due to increases in employee compensation and infras


In [12]:
# ARREGLO 2 (8 min) — Híbrido BM25 + denso.
#
# El argumento estándar, y es bueno: los embeddings entienden el significado
# pero se pierden con lo literal. «NVDA», «FY2025», «$91.4 billion» son
# cadenas exactas, y para eso una búsqueda léxica de los años setenta gana a
# cualquier transformer.
#
# BM25 va dado. Lo que hay que escribir es la FUSIÓN de las dos listas.
bm25, chunks_bm = miax_s2.montar_bm25()
POR_ID = {c["chunk_id"]: c for c in chunks_bm}
ORDEN_BM = {c["chunk_id"]: n for n, c in enumerate(chunks_bm)}  # fila de cada fragmento en get_scores


def hibrido(consulta: str, ticker=None, fiscal_year=None, item=None,
            k: int = K, kk: int = 60) -> list[dict]:
    """Fusiona el orden denso con el orden BM25 por RRF."""
    # TODO (alumno): la fusión.
    #
    # NO se pueden sumar las puntuaciones: una es un coseno entre -1 y 1 y la
    # otra un número sin escala fija. Se combinan las POSICIONES, que es lo
    # que hace Reciprocal Rank Fusion:
    #
    #     RRF(d) = suma sobre cada lista de  1 / (kk + posicion_de_d)
    #
    #   1. Orden denso: con_filtros(consulta, ..., k=indice.ntotal).
    #      Quedaos con el diccionario chunk_id -> posición (empezando en 1).
    #   2. Orden léxico: bm25.get_scores(miax_s2.tokenizar(consulta)) os da
    #      una puntuación por fragmento, en el orden de `chunks_bm`.
    #      Ordenad de mayor a menor y quedaos con chunk_id -> posición.
    #      Filtrad a los mismos fragmentos que dejó pasar el filtro denso.
    #   3. Para cada fragmento, sumad 1/(kk+posición) de cada lista. Si no
    #      aparece en una, usad una posición muy grande (10**6).
    #   4. Devolved los k de mayor RRF, como dicts de fragmento.
    densos = con_filtros(consulta, ticker, fiscal_year, item, k=indice.ntotal)
    pos_densa = {f["chunk_id"]: p for p, f in enumerate(densos, 1)}

    puntos_bm25 = bm25.get_scores(miax_s2.tokenizar(consulta))
    lexicos = sorted((c["chunk_id"] for c in chunks_bm if c["chunk_id"] in pos_densa),
                     key=lambda i: -puntos_bm25[ORDEN_BM[i]])
    pos_lexica = {i: p for p, i in enumerate(lexicos, 1)}

    rrf = {i: 1 / (kk + pos_densa.get(i, 10**6)) + 1 / (kk + pos_lexica.get(i, 10**6))
           for i in pos_densa}
    mejores = sorted(rrf, key=lambda i: -rrf[i])[:k]
    por_id_denso = {f["chunk_id"]: f for f in densos}
    return [{**por_id_denso[i], "puntuacion": rrf[i]} for i in mejores]


RESULTADOS["3 · + híbrido BM25"] = medir(
    lambda g: hibrido(g["pregunta"], g["ticker"], g["fiscal_year"],
                      g["item_esperado"]),
    "+ híbrido BM25 + denso")

+ híbrido BM25 + denso             recall@5 =  46.2%   fallan: ['of-001', 'of-002', 'of-004', 'of-015', 'of-016', 'of-018', 'of-019']


## El arreglo que no arregló nada

El híbrido no ha movido la aguja. Puede que incluso la haya movido hacia
abajo. Antes de tocar nada, la pregunta correcta es **por qué**, y la
respuesta está a la vista desde el principio:

> **Vuestras preguntas están en español. El corpus está en inglés.**

BM25 cuenta coincidencias de palabras. «¿Cuánto creció el revenue de
Microsoft?» y `Microsoft Cloud revenue increased 23% to $168.9 billion`
comparten exactamente dos cadenas: `microsoft` y `revenue`. Todo lo demás que
BM25 podría aprovechar —la morfología, los sinónimos, el contexto— no cruza la
frontera del idioma. El modelo de *embeddings* aguanta mejor porque está
entrenado sobre significado, pero también sufre: `bge-small-en-v1.5` es
**monolingüe inglés**, y lo dice en el nombre.

Y mirad la lista de las que fallan, no solo el porcentaje: **no es la misma
lista que antes**. El híbrido arregla alguna y rompe alguna otra, y acaba en
el mismo sitio. Un cambio que mueve qué preguntas fallan sin mover cuántas no
es neutro: es ruido con coste de mantenimiento.

Tres cosas que llevarse de aquí, y la tercera es la importante:

1. **Un arreglo correcto puede no servir.** RRF sobre BM25 está bien
   implementado, es lo que recomienda la literatura y aquí no aporta. No
   porque esté mal, sino porque el cuello de botella de este sistema está en
   otro sitio.
2. **El orden en que se prueban los arreglos cambia la conclusión.** Si el
   híbrido se mide *después* de arreglar el idioma, aporta algo. Medido antes,
   parece inútil. Los dos números son verdad y hay que decir cuál se está
   dando.
3. **Sin medir, esto no se ve.** Un equipo sin golden set habría metido el
   híbrido en producción, habría añadido complejidad, latencia y una
   dependencia más, y habría tenido la sensación de haber mejorado el sistema.

El cuello de botella es el idioma. Eso es lo que arregla el tercero.

In [13]:
# ARREGLO 3 — Reescritura de consulta con el propio LLM.
#
# La idea: antes de buscar, el modelo convierte la pregunta del usuario en una
# consulta pensada para el índice. En este corpus eso significa sobre todo
# TRADUCIRLA, y de paso cambiar el vocabulario coloquial por el del informe
# («cuánto creció» -> «revenue increased», «ingresos» -> «net sales»).
#
# Es el arreglo más caro de los tres: añade una llamada al modelo antes de
# cada búsqueda. Por eso se mide.
INSTRUCCION = """Reescribe esta pregunta como una consulta de búsqueda para un
índice de informes 10-K en INGLÉS. Usa el vocabulario del propio informe.
Devuelve SOLO la consulta, sin comillas ni explicación."""

reescritor = None
if HAY_CLAVE:
    try:
        from langchain.chat_models import init_chat_model
        reescritor = init_chat_model(MODELO, temperature=0)
    except Exception as e:
        print(f"Sin reescritor en vivo ({type(e).__name__}).")


def reescribir(pregunta: str, id_golden: str | None = None) -> str:
    """La pregunta, convertida en consulta. Cae a la grabada si no hay red."""
    if reescritor is not None:
        try:
            return reescritor.invoke(
                [{"role": "system", "content": INSTRUCCION},
                 {"role": "user", "content": pregunta}]).text.strip()
        except Exception:
            pass
    return miax_s2.REESCRITURAS_RESPALDO.get(id_golden or "", pregunta)


print("Ejemplos de reescritura:")
for g in con_ancla[:3]:
    print(f"  ES: {g['pregunta']}")
    print(f"  EN: {reescribir(g['pregunta'], g['id'])}\n")

RESULTADOS["4 · + reescritura de consulta"] = medir(
    lambda g: con_filtros(reescribir(g["pregunta"], g["id"]), g["ticker"],
                          g["fiscal_year"], g["item_esperado"]),
    "+ reescritura a inglés")

RESULTADOS["5 · reescritura + híbrido"] = medir(
    lambda g: hibrido(reescribir(g["pregunta"], g["id"]), g["ticker"],
                      g["fiscal_year"], g["item_esperado"]),
    "reescritura + híbrido")

Ejemplos de reescritura:
  ES: ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia en el mercado chino y los controles de exportación?


  EN: NVIDIA fiscal year 2025 Form 10-K China export controls licensing requirements competition domestic competitors

  ES: ¿Qué riesgo de seguridad asocia Microsoft en FY2025 al uso creciente de modelos de IA generativa en sus sistemas internos?


  EN: generative artificial intelligence AI security risks vulnerabilities internal systems operations data confidential information

  ES: ¿Qué dice Meta en FY2025 sobre las bases legales en las que se apoya para transferir datos de la Unión Europea a Estados Unidos?


  EN: Meta legal bases rely cross-border data transfers European Union United States Standard Contractual Clauses Data Privacy Framework



+ reescritura a inglés             recall@5 =  61.5%   fallan: ['of-001', 'of-004', 'of-015', 'of-018', 'of-020']


reescritura + híbrido              recall@5 =  69.2%   fallan: ['of-001', 'of-004', 'of-015', 'of-020']


In [14]:
# La tabla del bloque. Es una fila de vuestro informe del día 24.
#
# Y lleva una columna de coste, que no es decorativa: el arreglo que más sube
# el recall es también el único que añade una llamada al modelo por pregunta.
COSTES = {
    "1 · denso plano":             "0 llamadas al LLM",
    "2 · + filtro de metadatos":   "0 llamadas al LLM",
    "3 · + híbrido BM25":          "0 llamadas, +1 índice en memoria",
    "4 · + reescritura de consulta": "1 llamada al LLM por búsqueda",
    "5 · reescritura + híbrido":   "1 llamada + índice léxico",
}

tabla = pd.DataFrame(
    [{"configuración": k, "recall@5": f"{v:.1%}", "coste": COSTES.get(k, "")}
     for k, v in RESULTADOS.items()]
)
print(tabla.to_string(index=False))

mejor = max(RESULTADOS, key=RESULTADOS.get)
print(f"\nMejor: {mejor} ({RESULTADOS[mejor]:.1%}) frente al "
      f"{RESULTADOS['1 · denso plano']:.1%} de partida.")

# La que sigue fallando después de los tres arreglos merece medio minuto:
# es una pregunta sobre competencia en China cuya frase vive en un Item 1A de
# 19.476 tokens lleno de párrafos sobre controles de exportación. La consulta
# recupera los párrafos de controles de exportación, que es lo que ha pedido.
# Para esa hace falta otra cosa: reordenar con un cross-encoder, o subir k.
# No da tiempo hoy, y es un buen punto para el informe.

# --- verificación de §1 --------------------------------------------------
assert len(RESULTADOS) == 5, "Faltan configuraciones por medir."

sin_terminar = [k for k, v in RESULTADOS.items() if v == 0.0]
if sin_terminar:
    print("\nOJO: un 0 % clavado no es un mal resultado, es una función que "
          "devuelve None. Repasad el ejercicio de:")
    for k in sin_terminar:
        print(f"  - {k}")
else:
    assert (RESULTADOS["2 · + filtro de metadatos"]
            >= RESULTADOS["1 · denso plano"]), \
        ("El filtro de metadatos ha EMPEORADO el recall, y eso no puede "
         "pasar: filtrar solo quita candidatos que no son del documento por "
         "el que se pregunta. Revisad la comparación de fiscal_year, que en "
         "el índice es un entero.")
    print("\n§1 listo.")

                configuración recall@5                            coste
              1 · denso plano    30.8%                0 llamadas al LLM
    2 · + filtro de metadatos    46.2%                0 llamadas al LLM
           3 · + híbrido BM25    46.2% 0 llamadas, +1 índice en memoria
4 · + reescritura de consulta    61.5%    1 llamada al LLM por búsqueda
    5 · reescritura + híbrido    69.2%        1 llamada + índice léxico

Mejor: 5 · reescritura + híbrido (69.2%) frente al 30.8% de partida.

§1 listo.


## Por qué un RAG plano no contesta «qué cambió entre FY2024 y FY2025»

Acabáis de subir el recall del retrieval. Eso arregla dos de los cinco fallos
del diagnóstico. El tercero no se arregla recuperando mejor, por bien que se
recupere, y merece la pena ver exactamente por qué.

«¿Cómo varió el beneficio neto de Meta entre 2024 y 2025, y qué explica esa
variación?» necesita cuatro cosas:

1. el beneficio neto de FY2024,
2. el de FY2025,
3. la explicación, que está en la prosa del MD&A de FY2025,
4. y **restar**.

Un *pipeline* de recuperación única hace una consulta, trae *k* fragmentos y
genera. Puede traer los dos años mezclados y dejar que el modelo se apañe —y
el modelo se apaña, mal— pero no puede **decidir que hacen falta dos búsquedas
distintas**, porque esa decisión está fuera de su arquitectura: el número de
recuperaciones lo fijó quien escribió el *pipeline*, no la pregunta.

Aquí es donde el agente deja de ser decoración sobre una *pipeline*. No porque
sea más listo, sino porque **el número de pasos lo decide en tiempo de
ejecución**. Eso es todo lo que aporta, y es suficiente.

> Por esto el enunciado os exige **6 comparativas** entre vuestras 20
> preguntas. Sin ellas, un RAG plano aprueba vuestro golden set y el agente se
> queda sin justificar — que es precisamente lo que el curso quiere que
> descubráis midiendo, no que os creáis porque lo diga el profesor.

In [15]:
# La comparativa contra retrieval plano. Una consulta, k fragmentos, generar.
comparativa = next(g for g in golden if g["id"] == "of-020")
print(comparativa["pregunta"])
print(f"esperado: {comparativa['respuesta_esperada']}\n")

fragmentos = con_filtros(reescribir(comparativa["pregunta"],
                                    comparativa["id"]),
                         comparativa["ticker"],
                         comparativa["fiscal_year"]) or []
for f in fragmentos:
    print(f"  [{f['chunk_id']}] FY{f['fiscal_year']} item {f['item']} "
          f"· {f['puntuacion']:.3f}")

if HAY_CLAVE:
    from langchain.chat_models import init_chat_model
    plano = init_chat_model(MODELO, temperature=0)
    respuesta = plano.invoke([
        {"role": "system",
         "content": "Responde SOLO con los fragmentos dados. Si no están "
                    "los datos, dilo."},
        {"role": "user",
         "content": f"{comparativa['pregunta']}\n\n"
                    f"{miax_s2.formatear_fragmentos(fragmentos)}"},
    ]).text
    print(f"\nRAG PLANO:\n{respuesta}")

# Mirad qué falta: la cifra de FY2024 no está en ningún fragmento recuperado,
# porque la consulta iba de la variación y el filtro era del ejercicio 2025.
# El modelo puede describir la causa y no puede dar las dos cifras.

¿Cómo varió el beneficio neto de Meta entre 2024 y 2025, y qué explica esa variación?
esperado: Bajó de 62.360 millones de dólares a 60.458 millones de dólares (−3,1 %): la provisión por impuestos creció 17.170 millones (un 207 %) por el aumento del tipo efectivo.



  [META-2025-8-0010] FY2025 item 8 · 0.826
  [META-2025-8-0017] FY2025 item 8 · 0.821
  [META-2025-8-0009] FY2025 item 8 · 0.792
  [META-2025-8-0013] FY2025 item 8 · 0.784
  [META-2025-8-0000] FY2025 item 8 · 0.778



RAG PLANO:
Según los fragmentos proporcionados:

### Variación del beneficio neto (*Net income*)
Entre 2024 y 2025, el beneficio neto de Meta disminuyó en **$1.902 millones**:
* **2024:** $62.360 millones
* **2025:** $60.458 millones

---

### Factores que reflejan dicha variación
De acuerdo con el estado de resultados consolidado (*Consolidated Statements of Income* [META-2025-8-0009]):

1. **Aumento sustancial en la provisión de impuestos:** A pesar de que los ingresos antes de impuestos sobre la renta (*Income before provision for income taxes*) aumentaron de $70.663 millones en 2024 a $85.932 millones en 2025, la provisión para impuestos sobre la renta (*Provision for income taxes*) se incrementó de **$8.303 millones** en 2024 a **$25.474 millones** en 2025.
2. **Incremento de costos y gastos totales (*Total costs and expenses*):** Crecieron de $95.121 millones a $117.690 millones, impulsados por:
   * **Investigación y desarrollo (*Research and development*):** de $43.873 millone

In [16]:
# La misma pregunta con el agente. Lo que cambia no es el modelo: es que
# decide cuántas veces busca.
if origen is not None:
    r = responder(comparativa["pregunta"], thread_id="comparativa")
    miax_s2.pretty_trace(r)
    print(f"\n  herramientas: {miax_s2.herramientas_usadas(r)}")
    print(f"  [{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
else:
    print("Sin clave: este bloque no se puede ejecutar.")

# En la trayectoria tiene que verse get_xbrl_fact DOS veces, una por
# ejercicio, y una búsqueda de texto para la explicación. Si solo aparece una
# llamada, el agente no ha descompuesto la pregunta: ha contestado media.

OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


  1. list_available()
       -> AAPL · Apple Inc. FY2024: Items 1A, 7, 7A, 8 AAPL · Apple Inc. FY2025: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2024: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2025: Items 1A, 7, 7A, 8 GOOGL · Alphabet Inc. FY2024: It…
  2. get_xbrl_fact(fiscal_year=2024, concept='NetIncomeLoss', ticker='META')
       -> META FY2024 · NetIncomeLoss = 62360000000.0 USD (cierre 2024-12-31, 10-K)
  3. get_xbrl_fact(concept='NetIncomeLoss', fiscal_year=2025, ticker='META')
       -> META FY2025 · NetIncomeLoss = 60458000000.0 USD (cierre 2025-12-31, 10-K)
  4. search_filings(ticker='META', item='7', query='Net income decreased provision for income taxes income from operations 2025 2024', fiscal_year=2025)
       -> [META-2025-7-0028] META FY2025 Item 7 · Provision for income taxes (puesto 1 de 5) Provision for income taxes  Year Ended December 31, 2025		2024		2023		2025 vs 2024 % change		2024 vs 2023 % change (in millions, except p…
  5. RespuestaFinanciera(citas=[{'cita'

In [17]:
# Memoria entre turnos. `thread_id` es lo que convierte dos invocaciones
# sueltas en una conversación: el checkpointer guarda el estado y la segunda
# llamada llega con todo lo anterior delante.
#
# Sin esto, la pregunta de seguimiento no tiene antecedente y el modelo se
# inventa de qué le están hablando.
if origen is not None:
    seguimiento = "¿Y el de Microsoft en el mismo periodo?"
    r2 = responder(seguimiento, thread_id="comparativa")   # MISMO hilo
    miax_s2.pretty_trace(r2)

    r3 = responder(seguimiento, thread_id="hilo-nuevo")    # hilo distinto
    print("\n--- el mismo seguimiento, en un hilo sin historia ---")
    miax_s2.pretty_trace(r3)
else:
    print("Sin clave: este bloque no se puede ejecutar.")

# La segunda tiene que fallar o pedir aclaración. Si contesta igual de bien,
# es que el modelo ha adivinado, y adivinar no es memoria.

# --- verificación de §2 --------------------------------------------------
assert "of-020" in {g["id"] for g in golden}, \
    "El golden set cargado no es el oficial: §2 necesita la comparativa."
print("\n§2 listo.")

Deserializing unregistered type agente.interfaz.RespuestaFinanciera from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('agente.interfaz', 'RespuestaFinanciera')]


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


OpenRouter: firma de pensamiento corrupta; se reenvía sin razonamiento previo.


  1. list_available()
       -> AAPL · Apple Inc. FY2024: Items 1A, 7, 7A, 8 AAPL · Apple Inc. FY2025: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2024: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2025: Items 1A, 7, 7A, 8 GOOGL · Alphabet Inc. FY2024: It…
  2. get_xbrl_fact(fiscal_year=2024, concept='NetIncomeLoss', ticker='META')
       -> META FY2024 · NetIncomeLoss = 62360000000.0 USD (cierre 2024-12-31, 10-K)
  3. get_xbrl_fact(concept='NetIncomeLoss', fiscal_year=2025, ticker='META')
       -> META FY2025 · NetIncomeLoss = 60458000000.0 USD (cierre 2025-12-31, 10-K)
  4. search_filings(ticker='META', item='7', query='Net income decreased provision for income taxes income from operations 2025 2024', fiscal_year=2025)
       -> [META-2025-7-0028] META FY2025 Item 7 · Provision for income taxes (puesto 1 de 5) Provision for income taxes  Year Ended December 31, 2025		2024		2023		2025 vs 2024 % change		2024 vs 2023 % change (in millions, except p…
  5. RespuestaFinanciera(citas=[{'cita'


--- el mismo seguimiento, en un hilo sin historia ---
  1. list_available()
       -> AAPL · Apple Inc. FY2024: Items 1A, 7, 7A, 8 AAPL · Apple Inc. FY2025: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2024: Items 1A, 7, 7A, 8 AMZN · AMAZON COM INC FY2025: Items 1A, 7, 7A, 8 GOOGL · Alphabet Inc. FY2024: It…
  2. RespuestaFinanciera(respuesta='No es posible responder a la pregunta porque no se especifica la métrica o concepto financiero ni el ejercicio fiscal de referencia al que hace alusión ("el mismo periodo"). Por favor, indique la métrica (por ejemplo, ingresos, beneficio neto, etc.) y el ejercicio fiscal deseado para Microsoft.', fuente='ninguna', ticker='MSFT')
       -> Returning structured response: respuesta='No es posible responder a la pregunta porque no se especifica la métrica o concepto financiero ni el ejercicio fiscal de referencia al que hace alusión ("el mismo periodo"). Por …

  respuesta: No es posible responder a la pregunta porque no se especifica la métrica o con

## Conversaciones largas: `SummarizationMiddleware`

Un apunte sin ejercicio, porque en vuestro entregable no hace falta y en
cualquier sistema real sí.

El estado de una conversación crece con cada turno, y cada turno se reenvía
entero al modelo. Una trayectoria con seis llamadas a `read_section` mete
decenas de miles de tokens en la ventana y los paga en **cada** vuelta
siguiente. Llega un momento en que o se trunca o no cabe.

```python
from langchain.agents.middleware import SummarizationMiddleware

SummarizationMiddleware(
    model=MODELO,
    max_tokens_before_summary=8000,   # a partir de aquí, resume
    messages_to_keep=6,               # los últimos, literales
)
```

Resume los mensajes viejos y deja los recientes intactos. El precio es que lo
resumido ya no es recuperable literalmente: si vuestra evaluación necesita la
trayectoria completa —y la del día 24 la necesita, porque
`uso_la_tool_correcta` la lee— resumir a mitad de una invocación os borra la
prueba. Por eso va mencionado y no puesto.

## Middleware: dónde se mete el código propio dentro del bucle

El bucle que escribisteis el día 10 era este:

```text
mientras queden vueltas:
    respuesta = modelo(mensajes)
    si no pide herramientas: terminar
    ejecutar las herramientas y añadir los resultados
```

**El middleware son ganchos en los huecos de ese bucle.** Nada más. Cada uno
recibe el estado y puede leerlo, cambiarlo o cortar la ejecución:

| Gancho | Cuándo entra | Para qué sirve aquí |
| --- | --- | --- |
| `before_model` | Antes de cada llamada al modelo | Recortar contexto, inyectar instrucciones |
| `after_model` | Justo después de la respuesta | **Verificar lo que acaba de decir** |
| `wrap_tool_call` | Alrededor de cada herramienta | Límites, permisos, reintentos |
| `after_agent` | Al terminar del todo | Auditoría, registro |

Dos cosas que hay que tener claras antes de escribir uno:

1. **Se ejecutan en el orden de la lista**, y el orden importa: un límite de
   llamadas que va después del verificador no impide que el verificador se
   ejecute una vez de más.
2. **Añadir un mensaje NO hace que el agente vuelva a pensar.** Esta es la
   parte que sorprende, y conviene entenderla antes de escribir el ejercicio
   de la celda 23.

   Cuando `after_model` termina, el grafo mira el último `AIMessage` para
   decidir a dónde va. Si ese mensaje no pedía herramientas —el caso normal
   de una respuesta final— **el agente termina**, y da igual cuántos mensajes
   le hayáis añadido detrás: se quedan en el estado sin que nadie los lea.

   Para que haya otra vuelta hay que decirlo explícitamente, y son dos cosas a
   la vez:

   ```python
   @after_model(can_jump_to=["model"])        # se declara al decorar
   def mi_middleware(state, runtime):
       return {"messages": [...], "jump_to": "model"}   # y se pide al volver
   ```

   El `can_jump_to` construye la arista del grafo; el `jump_to` la usa. Sin el
   primero, el segundo no tiene por dónde ir.
3. **Y ahí está el riesgo:** un verificador que se queja siempre es un bucle
   infinito con otro nombre. Además, `ToolCallLimitMiddleware` **no** os
   protege de este, porque un modelo que responde sin llamar a ninguna
   herramienta no gasta llamadas a herramienta. Para eso está
   `ModelCallLimitMiddleware`, y para eso el verificador de la celda 23
   corrige **una sola vez**.

In [18]:
# El bucle infinito del día 10, cerrado.
#
# La pregunta era el margen bruto de Amazon, y el problema no era el modelo:
# Amazon NO etiqueta GrossProfit en us-gaap. La herramienta devuelve el mismo
# aviso una y otra vez y el modelo reintenta con variantes del concepto.
#
# `ToolCallLimitMiddleware` no le enseña nada al modelo: le pone un techo.
# Es control de coste, no inteligencia, y es la primera cosa que se pone.
from langchain.agents.middleware import ToolCallLimitMiddleware

PREGUNTA_BUCLE = ("¿Cuál fue el margen bruto de Amazon en 2025 y cómo cambió "
                  "respecto a 2024?")

if HAY_CLAVE:
    acotado = miax_s2.baseline(
        MODELO, middleware=[ToolCallLimitMiddleware(run_limit=8)])
    resultado, segundos = miax_s2.cronometrar(
        acotado.invoke,
        {"messages": [{"role": "user", "content": PREGUNTA_BUCLE}]},
        config={"configurable": {"thread_id": "limite"}},
    )
    miax_s2.pretty_trace(resultado)
    usadas = miax_s2.herramientas_usadas(resultado)
    print(f"\n  {len(usadas)} llamadas a herramienta · {segundos:.1f} s")
    print("  La semana pasada esto no paraba. Hoy para en 8.")
else:
    print("Sin clave: este bloque no se puede ejecutar.")

# Los tres comportamientos al llegar al límite, y cuál queréis:
#   "continue" (por defecto) bloquea la llamada y deja que el modelo siga con
#              lo que tiene. Es el que da una respuesta parcial en vez de una
#              excepción, y normalmente es el que queréis en el día 24.
#   "error"    lanza ToolCallLimitExceededError. Útil en pruebas.
#   "end"      termina la invocación en seco.
#
# Y hay dos límites distintos: run_limit (por invocación) y thread_limit (por
# conversación, exige checkpointer).
#
# OJO con lo que este límite NO cubre: cuenta llamadas a HERRAMIENTA. Un
# modelo que se enrosca contestando sin llamar a ninguna no gasta ni una, y
# este middleware lo deja correr. Para ese caso está su gemelo:
#
#     ModelCallLimitMiddleware(run_limit=10)
#
# Los dos juntos acotan las dos formas de no terminar. Aparece en la celda 24,
# en cuanto el verificador pueda provocar vueltas extra.

  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A'…
  2. get_xbrl_fact(concept='GrossProfit', ticker='AMZN', fiscal_year=2025)
  3. get_xbrl_fact(concept='Revenues', ticker='AMZN', fiscal_year=2025)
  4. get_xbrl_fact(ticker='AMZN', concept='CostOfGoodsAndServicesSold', fiscal_year=2025)
       -> AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'NetCashProvidedByUsedInOperatingActivities', 'NetIn…
       -> AMZN no reportó 'Revenues' en FY2025. Conceptos disponibles: ['Assets', 'CashAndCashEquivalentsAtCarryingValue', 'EarningsPerShareBasic', 'EarningsPerShareDiluted', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncom…
       -> AMZN no reportó 'CostOfGoodsAndServ

In [19]:
# EJERCICIO CENTRAL (15 min) — verificar_cifras_contra_xbrl
#
# Este es el middleware que hace que vuestro entregable sea de dominio
# financiero y no un chatbot con documentos. La idea cabe en una frase:
#
#   si la respuesta afirma un número, contrástalo con el XBRL antes de
#   dejarla salir; y si no cuadra, devuélvele el desajuste al modelo.
from langchain.agents.middleware import AgentState, after_model
from langgraph.runtime import Runtime

xbrl = pd.read_parquet(miax_s2.dir_corpus() / "xbrl_facts.parquet")
TOLERANCIA = 0.01          # 1 %: redondear no es inventarse un número
MARCA = "VERIFICACIÓN AUTOMÁTICA"


@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(state: AgentState,
                                 runtime: Runtime) -> dict | None:
    """Contrasta la cifra de la respuesta con el XBRL. Si no cuadra, se lo
    devuelve al modelo para que se corrija."""
    # TODO (alumno):
    #
    #   1. Sacad `structured_response` del estado. Si no hay, o si
    #      `.cifra` es None, devolved None: no hay nada que verificar.
    #      Devolver None significa "sigue, no toco nada".
    #   2. Necesitáis saber CONTRA QUÉ comparar: `.ticker` y `.ejercicio`
    #      de la respuesta estructurada. Si faltan, tampoco hay verificación.
    #   3. Una corrección por invocación, y esto NO es opcional: si ya hay un
    #      mensaje con `MARCA` en `state["messages"]`, devolved None. Sin
    #      ese freno, un modelo que insiste en su cifra os deja dando vueltas
    #      entre el modelo y el verificador, y el límite de llamadas a
    #      herramienta no lo corta porque no gasta ninguna.
    #   4. Filtrad `xbrl` por ese ticker y ese ejercicio.
    #   5. ¿Alguno de los hechos reportados cuadra con la cifra afirmada?
    #      `miax_s2.cuadra(afirmada, real, TOLERANCIA)` os lo dice.
    #      Si alguno cuadra: None (todo bien).
    #   6. Si ninguno cuadra, devolved el mensaje correctivo Y el salto:
    #
    #          return {"messages": [{"role": "user", "content": "..."}],
    #                  "jump_to": "model"}
    #
    #      El `jump_to` es la mitad que se olvida. Sin él, el mensaje se
    #      queda en el estado, el agente termina igual y la respuesta mala
    #      sale con una queja detrás que no lee nadie. Empezad por ahí: es
    #      el fallo que os va a costar más encontrar.
    #
    #      Y en el contenido, decidle QUÉ afirmó, QUÉ hay reportado de verdad
    #      y que lo corrija o diga que no está en el corpus. Un mensaje vago
    #      no corrige nada.
    ...


print("Middleware definido.")

Middleware definido.


In [20]:
# Probarlo. La pregunta es la del diagnóstico que se inventaba la cifra:
# Alphabet etiqueta `Revenues`, no el concepto largo que usa Apple.
trampa = next(g for g in golden if g["id"] == "of-012")
print(f"{trampa['pregunta']}\nesperado: {trampa['respuesta_esperada']}\n")

if HAY_CLAVE:
    from langchain.agents.middleware import ModelCallLimitMiddleware

    verificado = miax_s2.baseline(MODELO, middleware=[
        ToolCallLimitMiddleware(run_limit=8),     # los límites, PRIMERO
        ModelCallLimitMiddleware(run_limit=10),   # y este acota el bucle
        verificar_cifras_contra_xbrl,             # que puede abrir el de abajo
    ])
    r = verificado.invoke(
        {"messages": [{"role": "user", "content": trampa["pregunta"]}]},
        config={"configurable": {"thread_id": "verificado"}},
    )
    miax_s2.pretty_trace(r)

    afirmada = r["structured_response"].cifra
    print(f"\n  afirmada: {afirmada:,.0f}" if afirmada else "\n  sin cifra")
    print(f"  esperada: {trampa['cifra_esperada']:,.0f}")
    print(f"  cuadra: "
          f"{miax_s2.cuadra(afirmada or 0, trampa['cifra_esperada'])}")
else:
    print("Sin clave: este bloque no se puede ejecutar.")

# Si en la trayectoria aparece el mensaje de VERIFICACIÓN AUTOMÁTICA seguido
# de otra llamada a get_xbrl_fact, el guardrail hizo su trabajo: el modelo se
# corrigió solo y queda registrado que lo hizo.

¿Cuáles fueron los ingresos de Alphabet en 2025?
esperado: 402.836 millones de dólares.



  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A'…
  2. get_xbrl_fact(concept='Revenues', fiscal_year=2025, ticker='GOOGL')
       -> GOOGL FY2025 Revenues = 402,836,000,000 USD (cierre 2025-12-31, 10-K)
  3. RespuestaFinanciera(ejercicio=2025, unidad='USD', fuente='xbrl', respuesta='Los ingresos de Alphabet Inc. (GOOGL) en el ejercicio fiscal 2025 fueron de 402.836.000.000 USD (aproximadamente 402.836 millones de dólares).', ticker='GOOGL', cifra=402836000000)
       -> Returning structured response: respuesta='Los ingresos de Alphabet Inc. (GOOGL) en el ejercicio fiscal 2025 fueron de 402.836.000.000 USD (aproximadamente 402.836 millones de dólares).' cifra=402836000000.0 unidad='USD' …

  respuesta: Los ingresos de Alphabet Inc. (GOOGL) en el ejercicio fiscal 2025 fueron de 402.836.0

In [21]:
# HumanInTheLoopMiddleware: parar y preguntar, pero solo cuando toca.
#
# Un HITL que interrumpe en cada llamada no se usa: la gente aprueba sin leer
# a la tercera vez. El predicado `when` es lo que lo hace utilizable — se
# interrumpe en lo caro y en lo comprometido, y nada más.
from langchain.agents.middleware import HumanInTheLoopMiddleware


def es_seccion_muy_larga(peticion) -> bool:
    """read_section sobre un Item 1A de META son 34.751 tokens. Eso se
    aprueba a mano o no se aprueba."""
    args = peticion.tool_call["args"]
    for s in secciones:
        if (s["ticker"] == args.get("ticker")
                and s["fiscal_year"] == int(args.get("fiscal_year", 0))
                and s["item"] == args.get("item")):
            return s["n_tokens"] > 20000
    return False


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "read_section": {"allowed_decisions": ["approve", "reject"],
                         "when": es_seccion_muy_larga},
        "search_filings": False,          # nunca interrumpe
        "get_xbrl_fact": False,
    },
    description_prefix="Pendiente de aprobación",
)

if HAY_CLAVE:
    con_hitl = miax_s2.baseline(MODELO, middleware=[
        ToolCallLimitMiddleware(run_limit=8),
        hitl,
        verificar_cifras_contra_xbrl,
    ])
    CONFIG_HITL = {"configurable": {"thread_id": "hitl"}}
    estado = con_hitl.invoke(
        {"messages": [{"role": "user", "content":
                       "Léete entero el apartado de factores de riesgo de "
                       "Meta de 2025 y resúmelo."}]},
        config=CONFIG_HITL,
    )
    interrupciones = estado.get("__interrupt__")
    print("INTERRUMPIDO:" if interrupciones else "Terminó sin interrumpir:")
    print(interrupciones or estado.get("structured_response"))
else:
    print("Sin clave: este bloque no se puede ejecutar.")

# HITL exige checkpointer: sin él no hay dónde guardar la ejecución a medias,
# y sin eso no se puede reanudar.

Terminó sin interrumpir:
respuesta='En el informe anual 10-K correspondiente al ejercicio fiscal 2025 de Meta Platforms, Inc. (Item 1A), los factores de riesgo se estructuran en cinco grandes categorías principales:\n\n1. **Riesgos relacionados con la oferta de productos y publicidad:**\n   - **Compromiso y retención de usuarios:** Capacidad para retener y sumar usuarios frente a la competencia y cambios de preferencias.\n   - **Ingresos publicitarios y señales de datos:** Dependencia del gasto publicitario y pérdida o restricciones en señales de datos de terceros (por cambios de plataformas como iOS/Android y normativas de privacidad), lo que afecta la eficacia de la segmentación y medición de anuncios.\n   - **Plataformas móviles de terceros:** Dependencia del funcionamiento e interoperabilidad con sistemas operativos móviles (Apple y Google).\n\n2. **Inversiones en tecnologías emergentes, operaciones y resultados financieros:**\n   - **Inteligencia Artificial y Reality Labs:** Fuert

In [22]:
# Reanudar después de la interrupción.
#
# La invocación no se ha perdido: está guardada en el checkpointer, a medio
# camino. `Command(resume=...)` la retoma exactamente donde estaba.
from langgraph.types import Command

if HAY_CLAVE and 'interrupciones' in dir() and interrupciones:
    continuado = con_hitl.invoke(
        Command(resume=[{"type": "reject",
                         "message": "Demasiado caro. Usa search_filings."}]),
        config=CONFIG_HITL,
    )
    miax_s2.pretty_trace(continuado)
    print("\nEl rechazo vuelve al modelo como información, no como error: "
          "el agente busca otra forma de contestar.")
else:
    print("Nada que reanudar.")

# --- verificación de §3 --------------------------------------------------
assert callable(verificar_cifras_contra_xbrl) or True
assert miax_s2.cuadra(100.0, 100.4, 0.01), "La tolerancia debería aceptar un redondeo."
assert not miax_s2.cuadra(100.0, 150.0, 0.01), "Y rechazar un 50 % de desvío."
print("\n§3 listo.")

Nada que reanudar.

§3 listo.


## El golden set no es el trámite del final: es el activo

Media hora, y es la que decide vuestra nota. Todo lo anterior —el recall, los
guardrails, el límite de llamadas— son cambios cuyo valor **no se puede
afirmar sin esta sección**. Un sistema sin evaluación no es un sistema mejor
ni peor: es un sistema del que no se sabe nada.

Tres evaluadores, y no son intercambiables:

| Evaluador | Qué comprueba | Qué error caza |
| --- | --- | --- |
| `cita_correcta` | Que el `chunk_id` existe y su texto respalda lo que se afirma | Citas inventadas, o correctas por casualidad |
| `cifra_coincide_xbrl` | Que la cifra coincide con `xbrl_facts`, con tolerancia | Números leídos de la prosa o estimados |
| `uso_la_tool_correcta` | Que la trayectoria pasó por la herramienta que tocaba | **Acertar por el camino equivocado** |

El tercero es el que enseña la lección del curso, y es el que os va a doler:
una pregunta numérica que acierta el número sin haber pasado por
`get_xbrl_fact` **cuenta como fallo**. Parece injusto y no lo es. Ese acierto
no generaliza: salió bien porque la cifra estaba en un párrafo legible, y el
día que venga en una tabla partida —el 41 % de los fragmentos de este corpus—
saldrá mal y nadie se enterará.

Por eso `pretty_trace` no era una utilidad para depurar. Sin trayectoria, el
tercer evaluador no se puede escribir.

In [23]:
# LOS TRES EVALUADORES (12 min). Firma común: el ítem del golden set y lo que
# devolvió el agente. Devuelven True, False o None (no aplica).


def cita_correcta(item: dict, resultado: dict) -> bool | None:
    """El chunk_id citado existe y su texto contiene lo que dice la cita."""
    # TODO (alumno):
    #   1. `resultado["structured_response"]` os da la RespuestaFinanciera.
    #   2. Sin chunk_id: False en las extractivas, None en las numéricas
    #      (no aplica).
    #   3. ¿Existe ese chunk_id? `POR_ID` es el diccionario del corpus.
    #      Si no existe, se lo inventó: False.
    #   4. ¿El texto del fragmento respalda la cita? Comparad normalizando
    #      con `miax_s2.normalizar`, y no exijáis la cita entera: el modelo
    #      recorta. Los primeros ~120 caracteres bastan.
    ...


def cifra_coincide_xbrl(item: dict, resultado: dict) -> bool | None:
    """La cifra afirmada coincide con la esperada, con tolerancia."""
    # TODO (alumno):
    #   1. Si el ítem no tiene `cifra_esperada`, esto no aplica: None.
    #   2. Si el agente no devolvió cifra, es un fallo: False.
    #   3. Comparad con `miax_s2.cuadra(afirmada, esperada, TOLERANCIA)`.
    #      No uséis ==: 281.700 millones y 281.724 millones son la misma
    #      respuesta redondeada, y 250.000 no lo es.
    ...


def uso_la_tool_correcta(item: dict, resultado: dict) -> bool:
    """La trayectoria pasó por TODAS las herramientas esperadas."""
    # TODO (alumno):
    #   `miax_s2.herramientas_usadas(resultado)` os da la lista de nombres
    #   que aparecen en la trayectoria. El ítem trae `herramienta_esperada`.
    #   ¿Están todas las esperadas entre las usadas?
    #
    #   Y pensad qué implica: una pregunta numérica que acierta el número
    #   SIN pasar por get_xbrl_fact suspende aquí. Es lo que se pretende.
    ...


EVALUADORES = {
    "cita": cita_correcta,
    "cifra": cifra_coincide_xbrl,
    "trayectoria": uso_la_tool_correcta,
}
print("Evaluadores definidos:", list(EVALUADORES))

Evaluadores definidos: ['cita', 'cifra', 'trayectoria']


In [24]:
# `evaluar()`: el bucle sobre el JSONL. Esto va dado, y es literalmente lo que
# ejecutaréis el día 24 sobre `holdout.jsonl` — con VUESTRO `responder`.
#
# Que sea una función y no una celda no es cosmética: el §6 del enunciado os
# pide `evaluar("holdout.jsonl")` funcionando sobre un clon limpio. Si para
# ejecutarla hay que abrir un notebook y tocar cosas, los 20 minutos del día
# 24 no os llegan.
import time


def evaluar(preguntas: list[dict], funcion_responder, etiqueta: str = "",
            salida: str | None = None) -> pd.DataFrame:
    """Ejecuta el agente sobre las preguntas y aplica los tres evaluadores."""
    filas = []
    for i, item in enumerate(preguntas, 1):
        print(f"  [{i}/{len(preguntas)}] {item['id']}", end="\r")
        fila = {"id": item["id"], "familia": item["familia"],
                "ticker": item["ticker"]}
        try:
            comienzo = time.perf_counter()
            r = funcion_responder(item["pregunta"],
                                  thread_id=f"{etiqueta}-{item['id']}")
            fila["latencia_s"] = r.get("latencia_s",
                                       time.perf_counter() - comienzo)
            fila["coste_usd"] = r.get("coste_usd", 0.0)
            fila["llamadas"] = len(miax_s2.herramientas_usadas(r))
            for nombre, evaluador in EVALUADORES.items():
                fila[nombre] = evaluador(item, r)
            fila["recall"] = None          # se rellena abajo si hay ancla
            if item.get("ancla_texto"):
                fragmentos = con_filtros(
                    reescribir(item["pregunta"], item["id"]),
                    item["ticker"], item["fiscal_year"],
                    item["item_esperado"]) or []
                fila["recall"] = miax_s2.acierta(item, fragmentos)
        except Exception as e:
            fila["error"] = f"{type(e).__name__}: {e}"
        filas.append(fila)

    tabla = pd.DataFrame(filas)
    if salida:
        tabla.to_csv(salida, index=False)
    print(" " * 40, end="\r")
    return tabla


def resumir(tabla: pd.DataFrame, etiqueta: str) -> dict:
    """La fila de la tabla del informe."""
    def tasa(columna):
        valores = tabla[columna].dropna() if columna in tabla else []
        return float(valores.mean()) if len(valores) else float("nan")

    return {
        "versión": etiqueta,
        "cita": tasa("cita"),
        "cifra": tasa("cifra"),
        "trayectoria": tasa("trayectoria"),
        "recall@5": tasa("recall"),
        "coste medio (¢)": (tabla["coste_usd"].mean() * 100
                            if "coste_usd" in tabla else float("nan")),
        "latencia media (s)": (tabla["latencia_s"].mean()
                               if "latencia_s" in tabla else float("nan")),
        "llamadas/pregunta": (tabla["llamadas"].mean()
                              if "llamadas" in tabla else float("nan")),
    }


print("evaluar() y resumir() listos.")

evaluar() y resumir() listos.


In [25]:
# La tabla final: baseline contra final, sobre las mismas preguntas.
#
# Es la tabla que va en vuestro informe, y la pregunta de la defensa del día
# 24 es una sola: ¿las mejoras mejoraron algo de verdad, y a qué coste?
#
# En clase se ejecuta sobre las cinco duras para que quepa en el tiempo. En
# casa, sobre las 20 — y sobre vuestras 20, que es lo que se entrega.
if origen is not None:
    muestra = miax_s2.preguntas_duras(golden)

    agente_final = miax_s2.baseline(MODELO, middleware=[
        ToolCallLimitMiddleware(run_limit=8),
        ModelCallLimitMiddleware(run_limit=10),
        verificar_cifras_contra_xbrl,
    ])

    def responder_final(pregunta, thread_id=None):
        resultado, segundos = miax_s2.cronometrar(
            agente_final.invoke,
            {"messages": [{"role": "user", "content": pregunta}]},
            config={"configurable": {"thread_id": thread_id or "final"}},
        )
        return {**resultado,
                "coste_usd": miax_s2.coste_de(resultado, MODELO),
                "latencia_s": segundos}

    print("Baseline...")
    t_base = evaluar(muestra, responder, "base", "resultados_baseline.csv")
    print("Final...")
    t_fin = evaluar(muestra, responder_final, "fin", "resultados_final.csv")

    comparacion = pd.DataFrame([resumir(t_base, "baseline"),
                                resumir(t_fin, "final")])
    print(comparacion.round(3).to_string(index=False))
    print("\nPor pregunta, la versión final:")
    print(t_fin.to_string(index=False))
else:
    print("Sin clave: la tabla no se puede generar hoy. El código queda "
          "escrito y se ejecuta en casa.")

# --- verificación de §4 --------------------------------------------------
assert set(EVALUADORES) == {"cita", "cifra", "trayectoria"}, \
    "Faltan evaluadores."
assert cifra_coincide_xbrl({"cifra_esperada": None}, {}) is None, \
    "Una pregunta sin cifra esperada no aplica al evaluador de cifras."
print("\n§4 listo.")

Baseline...


Final...


 versión  cita  cifra  trayectoria  recall@5  coste medio (¢)  latencia media (s)  llamadas/pregunta
baseline   NaN    NaN          NaN      0.50            2.572              41.487                5.0
   final   NaN    NaN          NaN      0.75            0.948              15.717                4.0

Por pregunta, la versión final:
    id     familia ticker  latencia_s  coste_usd  llamadas cita cifra trayectoria recall
of-006  extractiva   AMZN   13.881088   0.010158         4 None  None        None   True
of-002  extractiva   MSFT    9.817701   0.006511         3 None  None        None   True
of-020 comparativa   META   26.825160   0.014113         5 None  None        None  False
of-012    numerica  GOOGL    8.085779   0.004401         3 None  None        None   None
of-019 comparativa   AAPL   19.975409   0.012211         5 None  None        None   True

§4 listo.


## El día 24, y cómo funciona el hold-out

**Antes de las 0:00 del 24** se cierra la entrega: repositorio, golden set de
20 preguntas vuestras con al menos 6 comparativas, e informe con la tabla
*baseline* contra final.

**El 24, en clase**, recibís `holdout.jsonl`: 10 preguntas que no ha visto
nadie, con el mismo esquema que vuestro golden set. Tenéis **20 minutos** para
ejecutarlas. Por eso el §6 del enunciado os pide `evaluar("holdout.jsonl")`
funcionando sobre un clon limpio del repositorio: si hay que abrir un notebook
y tocar cosas, no llegáis.

**Probadlo antes del 23.** Clon limpio, ZIP recién subidos, `evaluar()` sobre
vuestro propio golden set. Cinco minutos, y es el ensayo general.

Dos cosas sobre el hold-out que conviene saber de antemano:

- **Al menos dos de las diez no tienen respuesta en el corpus.** Un ejercicio
  que no está, una magnitud que la compañía no reporta. Que el agente conteste
  `fuente="ninguna"` en vez de inventarse una cifra es la mitad del examen, y
  el corpus está lleno de sitios donde probarlo: Amazon no etiqueta
  `GrossProfit` ni `Liabilities`; Alphabet no usa el mismo concepto de
  ingresos que Apple.
- **Si el hold-out sale peor que vuestro golden set, eso no es un suspenso: es
  el hallazgo.** Sobreajustar al conjunto con el que has iterado es
  exactamente el error que se comete en producción. Verlo medido, en público y
  una vez, vale más que tres clases sobre validación. Defendedlo.

## Qué os lleváis de hoy

- **Medir antes de arreglar.** Habéis visto un arreglo correcto, recomendado y
  bien implementado que en este corpus no aportó nada. Sin golden set, habría
  entrado en producción con la sensación de haber mejorado el sistema.
- **El cuello de botella casi nunca está donde se mira primero.** Aquí era el
  idioma, no el algoritmo de recuperación.
- **Un middleware es un gancho en el bucle que ya escribisteis.** No hay magia:
  `after_model` es «después de la respuesta», y devolver mensajes es «otra
  vuelta».
- **La trayectoria es parte de la respuesta.** Acertar por el camino
  equivocado es un fallo, porque no generaliza. Ese es el evaluador que
  distingue vuestro sistema de uno que tuvo suerte.
- **Coste y latencia son columnas de la tabla**, no una nota al pie. La
  pregunta del día 24 no es si mejorasteis: es si compensó.

## Para el 24

| Qué | Cuándo |
| --- | --- |
| Repositorio con `responder()` y `evaluar()` sobre un clon limpio | Antes de las 0:00 del 24 |
| Golden set: 20 preguntas propias, ≥6 comparativas | Con el repositorio |
| Informe con la tabla baseline contra final (calidad **y** coste) | Con el repositorio |
| Presentación de 8 min con el resultado del hold-out | El 24, en clase |

El viernes 18 os explican ReAct y el paper; el 19, MCP, ADK y A2A. Hay un
notebook de bonus sobre MCP para después de esa clase: expone estas mismas
cuatro herramientas como servidor MCP, y el punto es que **el agente no
cambia**.